# Ampliación: RAG con LangChain

Este notebook repite el flujo del Lab 01 sin usar `learning_ai_rag`. La idea es ver qué aporta LangChain cuando seguimos tomando las decisiones importantes: fuentes, tamaño de chunk, evidencia, citas y abstención.

In [ ]:
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'datos-demo').exists())
KNOWLEDGE_DIR = ROOT / 'datos-demo' / 'knowledge'


## 1. Documentos Markdown

Leemos las mismas políticas que en el Lab 01 y las convertimos directamente en `Document` de LangChain. Cada documento guarda texto y metadatos: fuente y título.

In [ ]:
import re
from langchain_core.documents import Document

documents = []
for path in sorted(KNOWLEDGE_DIR.glob('*.md')):
    raw_text = path.read_text(encoding='utf-8').strip()
    title_match = re.search(r'^#\s+(.+)$', raw_text, flags=re.MULTILINE)
    title = title_match.group(1) if title_match else path.stem
    text = re.sub(r'^#\s+.+$', '', raw_text, count=1, flags=re.MULTILINE).strip()
    documents.append(Document(page_content=text, metadata={'source': path.name, 'title': title}))

print(f'{len(documents)} documentos cargados')
print(documents[0].metadata)


## 2. Chunking con LangChain

`RecursiveCharacterTextSplitter` intenta conservar bloques de texto naturales antes de dividirlos. Contamos palabras en lugar de caracteres para mantener el mismo objetivo docente del Lab 01: chunks de 120 palabras con 30 de solapamiento. Después asignamos un identificador estable para poder citar cada fragmento.

In [ ]:
from collections import defaultdict
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=30,
    length_function=lambda text: len(text.split()),
    separators=['\n\n', '\n', ' ', ''],
)
chunks = splitter.split_documents(documents)
chunk_number = defaultdict(int)
for chunk in chunks:
    source = chunk.metadata['source']
    chunk_number[source] += 1
    chunk.metadata['chunk_id'] = f'{source}#{chunk_number[source]}'

print(f'{len(chunks)} chunks creados')
print(chunks[0].metadata)
print(chunks[0].page_content[:300])


## 3. Embeddings y almacén en memoria

`HuggingFaceEmbeddings` ejecuta localmente el mismo modelo MiniLM-L12-v2. `InMemoryVectorStore` guarda los embeddings sólo mientras vive este kernel: no persiste nada en IRIS ni en disco.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    encode_kwargs={'normalize_embeddings': True},
)
store = InMemoryVectorStore(embedding=embeddings)
store.add_documents(chunks)
print('Almacén en memoria preparado.')


## 4. Búsqueda vectorial, evidencia y citas

La búsqueda devuelve fragments con una puntuación de similitud. La puntuación no es una respuesta: la usamos como señal para decidir si hay evidencia suficiente.

In [ ]:
question = '¿Cuándo puedo devolver un producto?'
results = store.similarity_search_with_score(question, k=3)

for document, score in results:
    print(f"[{document.metadata['chunk_id']}] {document.metadata['source']} — puntuación: {score:.3f}")
    print(document.page_content)
    print()


## 5. Contexto explícito y abstención

Aquí está la frontera entre búsqueda y RAG: reunimos los fragments recuperados en un contexto trazable. Si no superan el umbral, nos abstenemos antes de llamar a un modelo.

In [ ]:
MIN_SCORE = 0.35
evidence = [(document, score) for document, score in results if score >= MIN_SCORE]

if not evidence:
    context = ''
    print('No encuentro evidencia suficiente. Prefiero no inventar una respuesta.')
else:
    context = '\n\n'.join(
        f"[{document.metadata['chunk_id']}] Fuente: {document.metadata['source']}\n{document.page_content}"
        for document, _ in evidence
    )
    print(context)


## 6. Generación RAG opcional

La generación cloud es el último paso y es opcional. Sólo se ejecuta si activas `GENERATE` y existe `OPENROUTER_API_KEY`. Fíjate en que el prompt obliga a usar únicamente el contexto que acabamos de construir y a conservar sus citas.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(ROOT / '.env')
GENERATE = True  # Cambia a True sólo si quieres usar OpenRouter.

if GENERATE and evidence and os.getenv('OPENROUTER_API_KEY'):
    from langchain_openai import ChatOpenAI

    model = ChatOpenAI(
        model=os.getenv('OPENROUTER_MODEL') or 'openai/gpt-oss-20b',
        api_key=os.environ['OPENROUTER_API_KEY'],
        base_url='https://openrouter.ai/api/v1',
        temperature=0,
    )
    response = model.invoke(
        'Responde en español sólo con este contexto. Conserva las citas [archivo#chunk] relevantes. '
        'Si no basta, dilo.\n\n'
        f'Pregunta: {question}\n\nContexto:\n{context}'
    )
    print(response.content)
elif not evidence:
    print('No se llama al modelo porque falta evidencia suficiente.')
else:
    print('Recuperación completada. Activa GENERATE y configura OPENROUTER_API_KEY para añadir la respuesta RAG opcional.')
